Imports

In [1]:
# imports --> EDSL data might not be needed (yet)
#from esdl import esdl
#from esdl.esdl_handler import EnergySystemHandler
import pandas as pd
import numpy as np
import numpy_financial as npf
from decimal import Decimal, ROUND_HALF_UP
#from EL_input_data import electrolyser_capacity

#get profile data
from Prices_input_data import electricity_price_profiles_2030, electricity_price_profiles_2040, electricity_price_profiles_2050, hydrogen_price_profiles_2030, hydrogen_price_profiles_2040, hydrogen_price_profiles_2050, CF_electrolyser_profiles_2030, CF_electrolyser_profiles_2040, CF_electrolyser_profiles_2050, df_wind_profiles_HUBN, df_wind_profiles_HUBE, df_wind_profiles_HUBW, df_solar_profiles_NED


Input data

In [2]:
start_year = 2030                                                           # start year of all the business cases being operational, original value = 2030
windfarm_capacity = 700                                                     # MW, based on TNVDW, original value = 700
windfarm_location_hub = 'HUBE'                                                 # choose between 'HUBN', 'HUBE' or 'HUBW'  , original value = HUBE                        
solar_capacity = 000                                                             # MW, dummy number, original number =200
electricity_grid_capacity = 200                                                # MW, original number = 200
electrolyser_capacity = 500                                                    # MW, original number=500
min_load_electrolyser = electrolyser_capacity *0.1                             # MW
total_energy_consumption = 55.7                                                # KWh/kgH2
lhv_h2_kwh_kg = 33.33 
h2_storage_tariff_list = [3.24, 4.32, 5.40]                                    # EUR/MWh                                                          # kWh/kgH2
efficiency_electrolyser_LHV = lhv_h2_kwh_kg / total_energy_consumption         # efficiency electrolyser LHV in %
operational_mode_electrolyser = 'wind_following'                               # choose between 'market_following' or 'wind_following'  


#storage parameters (not used in this file yet. potentially in future storage analysis can be made)
storage_volume_capacity = 2.948                         # TWh, capacity of hydrogen that can be stored in the facility at a single moment in time  # original value = 3.115
storage_injection_capacity = 18.24/24*3                    # GW  # original value= 18.24/24*3
storage_production_capacity = 24/24*3                     # GW  #original value = 24/24*3
storage_type = 'salt_cavern'                        # choose between'salt_cavern' or 'depleted_gasfield' 
storage_location = 'offshore'                       # choose between 'onshore' and 'offshore'
structure_location = 'onshore'                     # choose between 'island', 'platform', 'onshore'
number_of_cycles = 3.8                           # original value = 3.6
hydrogen_stored = storage_volume_capacity*1000000*number_of_cycles                 # MWh, hydrogen throughput storage by could be overwritten by this operational analysis
hydrogen_in_storage_sum = hydrogen_stored *8760/number_of_cycles     # MWh, total throughput multiplied by the average storage duration in hours
#injection period?
#production period? etc.

#Note3: this calculation assumes that the electrolyser operator with purchase storage capacity. It could be considered to make this optional.
#Note4: current operational modes do not include check whether electricity is green or not and what this means for selling GoO. Wind following operational mode does not consider that additional electricity is purchased from the market 
#Note5: we can make a check such that the storage injection and production capacities are always sufficient to store to total 'hydrogen stored'7

Function to create profiles

In [3]:
#Set the right windprofile for the right location
if windfarm_location_hub == 'HUBN':
    windprofiles = df_wind_profiles_HUBN
elif windfarm_location_hub == 'HUBE':
    windprofiles = df_wind_profiles_HUBE
elif windfarm_location_hub == 'HUBW':
    windprofiles = df_wind_profiles_HUBW
else:
    print("Error: please select a suitable windfarm hub location, choose between 'HUBN', 'HUBE' or 'HUBW'.")
    raise SystemExit("Stopping the program") #stop program if no suitable location has been chosen

def sum_scenario(hourly_cashflow_scenarios): # sum the totals for a cash flow stream or operational profiles
        sum_pessimistic = hourly_cashflow_scenarios['Pessimistic'].sum()
        sum_mostlikely = hourly_cashflow_scenarios['Most likely'].sum()
        sum_optimistic = hourly_cashflow_scenarios['Optimistic'].sum()
        sum_cashflow_scenarios = pd.DataFrame({'Pessimistic': [sum_pessimistic], 'Most likely': [sum_mostlikely], 'Optimistic': [sum_optimistic]})
        return sum_cashflow_scenarios

def OWF_OEL_operation(windfarm_capacity, electrolyser_capacity, min_load_electrolyser, solar_capacity, egrid_capacity, windprofiles, solarprofiles, CF_electrolyser_profiles_decade, electricity_price_profiles_decade, hydrogen_price_profiles_decade, efficiency_electrolyser, h2_storage_tariff_list, operational_mode_electrolyser):
    windfarm_th_output = windprofiles * windfarm_capacity                               # is output in MWh
    solar_th_output = solarprofiles * solar_capacity                                    # is output in MWh
    electrolyser_th_input = CF_electrolyser_profiles_decade * electrolyser_capacity     # is output in MWh
    electricity_sold_market = windfarm_th_output * 0                                    # create empty df with zeros of the right format
    electricity_sold_PPA = windfarm_th_output * 0                                       # create empty df with zeros of the right format
    electricity_purchased_market = windfarm_th_output * 0                               # create empty df with zeros of the right format
    electricity_purchased_PPA = windfarm_th_output * 0                                  # create empty df with zeros of the right format
    electricity_not_purchased_market = windfarm_th_output * 0
    electricity_curtailment = windfarm_th_output * 0
    electricity_shortage = windfarm_th_output * 0
    electricity_shortage_hours = windfarm_th_output * 0
    for j,row in windfarm_th_output.iterrows(): 
        for i in windfarm_th_output.columns:
            if operational_mode_electrolyser == 'market_following': #the following is programmed to operate the electrolyser either market following or wind following                                                  # for all the hours in every scenario
                
                if electrolyser_th_input.loc[j,i] < min_load_electrolyser:
                    electrolyser_th_input.loc[j,i] = min_load_electrolyser      # electrolyser should be running on minimum load
                
                if windfarm_th_output.loc[j,i] + solar_th_output.loc[j,i] < electrolyser_th_input.loc[j,i]:                                                #check if there is more consumed by electrolyser than generated by the windfarm
                    electricity_purchased_market.loc[j,i] = electrolyser_th_input.loc[j,i] - windfarm_th_output.loc[j,i] - solar_th_output.loc[j,i]       #if that is the case electricity should be purchased from the market by the electrolyser
                    electricity_purchased_PPA.loc[j,i] = electrolyser_th_input.loc[j,i] - electricity_purchased_market.loc[j,i] #all the electricity that the windfarm generates is consumed via the PPA
                    #if electricity_purchased_market.loc[j,i] + electricity_purchased_PPA.loc[j,i] < min_load_electrolyser:
                    #    electricity_purchased_market.loc[j,i] = min_load_electrolyser - electricity_purchased_PPA.loc[j,i]
                    electricity_sold_PPA.loc[j,i] = electricity_purchased_PPA.loc[j,i]                                          #obviously, the electricity sold via the PPA is equal to the electricity purchased by the PPA
                    electricity_sold_market.loc[j,i] = 0                                                                        #since there is more electricity demand from the electrolyser than generated by the windfarm, the windfarm is not able to sell additional electricity to the market
                    
                    if abs(electricity_purchased_market.loc[j,i] - electricity_sold_market.loc[j,i]) > egrid_capacity:
                        electricity_not_purchased_market.loc[j,i] = electricity_purchased_market.loc[j,i] - egrid_capacity      # just tracks the electricity that can't be purchased
                        electricity_purchased_market.loc[j,i] = egrid_capacity                                                   #sets the electricity purchased from market to the maximum allowed by egrid capacity
                        
                        if electricity_purchased_market.loc[j,i] + electricity_purchased_PPA.loc[j,i] < min_load_electrolyser:
                            electricity_shortage.loc[j,i] = min_load_electrolyser - (electricity_purchased_market.loc[j,i] + electricity_purchased_PPA.loc[j,i]) #it could be that minimum load requirement is not met due to grid capacity limitation. This variable tracks the shortage of the electrolyser
                            electricity_shortage_hours.loc[j,i] =+ 1
                
                else:
                    #if electrolyser_th_input.loc[j,i] < min_load_electrolyser:
                    #    electricity_purchased_market.loc[j,i] = min_load_electrolyser - electrolyser_th_input.loc[j,i]
                    electricity_sold_market.loc[j,i] = windfarm_th_output.loc[j,i] + solar_th_output.loc[j,i] - electrolyser_th_input.loc[j,i] #Because the WF is generating more than the EL, remaining electricity is sold to the market
                    electricity_sold_PPA.loc[j,i] = electrolyser_th_input.loc[j,i]                                  #Main part is going to electrolyser
                    electricity_purchased_PPA.loc[j,i] = electricity_sold_PPA.loc[j,i]                              #PPA purchased is equal to PPA sold
                    electricity_purchased_market.loc[j,i] = 0                                                       #No additional electricity is purchased from the market by electrolyser
                    #if electricity_purchased_market.loc[j,i] + electricity_purchased_PPA.loc[j,i] < min_load_electrolyser:
                    #    electricity_purchased_market.loc[j,i] = min_load_electrolyser - electricity_purchased_PPA.loc[j,i] #if minimum load is not met, than still there has to be purchased some electriicty from the market
                    
                    if electricity_sold_market.loc[j,i] > egrid_capacity:
                        electricity_curtailment.loc[j,i] = electricity_sold_market.loc[j,i] - egrid_capacity        # electricity supply that exceeds demand and grid capacity had to be curtailed
                        electricity_sold_market.loc[j,i] = egrid_capacity                                           # electricity can be sold to market only if there is enough grid capacity
                        #if electricity_purchased_market.loc[j,i] + electricity_purchased_PPA.loc[j,i] < min_load_electrolyser:
                        #    electricity_shortage.loc[j,i] = min_load_electrolyser - (electricity_purchased_market.loc[j,i] + electricity_purchased_PPA.loc[j,i]) #it could be that minimum load requirement is not met due to grid capacity limitation. This variable tracks the shortage of the electrolyser                          
            
            elif operational_mode_electrolyser == 'wind_following':
                
                if windfarm_th_output.loc[j,i] + solar_th_output.loc[j,i] > electrolyser_capacity:             # if WF produces more than capacity of electrolyer, E first goes to OEL and remaining is sold to market
                    electricity_sold_PPA.loc[j,i] = electrolyser_capacity                                   #Maximum E sold to electrolyser
                    electricity_sold_market.loc[j,i] = windfarm_th_output.loc[j,i] + solar_th_output.loc[j,i] - electrolyser_capacity  #Remaining sold to market
                    electricity_purchased_PPA.loc[j,i] = electricity_sold_PPA.loc[j,i]                      #Selling PPA WF equals purchasing PPA electrolyser
                    
                    if electricity_sold_market.loc[j,i] > egrid_capacity:
                        electricity_curtailment.loc[j,i] = electricity_sold_market.loc[j,i] - egrid_capacity        # electricity supply that exceeds demand and grid capacity had to be curtailed
                        electricity_sold_market.loc[j,i] = egrid_capacity                                           # electricity can be sold to market only if there is enough grid capacity
                
                else:
                    electricity_sold_PPA.loc[j,i] = windfarm_th_output.loc[j,i] + solar_th_output.loc[j,i]
                    electricity_purchased_PPA.loc[j,i] = electricity_sold_PPA.loc[j,i]
                    
                    if electricity_purchased_PPA.loc[j,i] < min_load_electrolyser:
                        electricity_purchased_market.loc[j,i] = min_load_electrolyser - electricity_purchased_PPA.loc[j,i] # if wind following can't provide the required baseload, still electricity is purchased from the market
                        
                        if electricity_purchased_market.loc[j,i] > egrid_capacity:
                            electricity_shortage.loc[j,i] = electricity_purchased_market.loc[j,i] - egrid_capacity        # if there is not enough grid capacity to supply baseload for ELY, then there is shortage
                            electricity_purchased_market.loc[j,i] = egrid_capacity                                           # electricity purchase is adjusted based on the max grid capacity available
                            electricity_shortage_hours.loc[j,i] =+ 1
            
            else:
                print("Error: please select a suitable electrolyser operational mode, choose between 'market_following' or 'wind_following'.")
                raise SystemExit("Stopping the program") #stop program if no suitable operational mode has been chosen
    
    
    #Compute the total revenues and costs per cash flow stream
    revenues_electricity_sold_market_hourly = electricity_sold_market * electricity_price_profiles_decade
    revenues_electricity_sold_market_hourly_W = electricity_sold_market * electricity_price_profiles_decade * (windfarm_th_output/(windfarm_th_output+solar_th_output))
    revenues_electricity_sold_market_hourly_S = electricity_sold_market * electricity_price_profiles_decade * (solar_th_output/(windfarm_th_output+solar_th_output))
    revenues_electricity_sold_PPA_hourly = electricity_sold_PPA * electricity_price_profiles_decade
    revenues_electricity_sold_PPA_hourly_W = electricity_sold_PPA * electricity_price_profiles_decade * (windfarm_th_output/(windfarm_th_output+solar_th_output))
    revenues_electricity_sold_PPA_hourly_S = electricity_sold_PPA * electricity_price_profiles_decade * (solar_th_output/(windfarm_th_output+solar_th_output))
    costs_electricity_purchased_market_hourly = electricity_purchased_market * electricity_price_profiles_decade
    costs_electricity_purchased_PPA_hourly = electricity_purchased_PPA * electricity_price_profiles_decade
    electrolyser_production_profile = (electricity_purchased_market + electricity_purchased_PPA) * efficiency_electrolyser #hydrogen output in MWh/hour for every hour
    revenues_hydrogen_sold_PPA_hourly_without_storage = electrolyser_production_profile * hydrogen_price_profiles_decade
    hydrogen_baseload_price = (sum_scenario(hydrogen_price_profiles_decade) / 8760)
    revenues_hydrogen_sold_PPA_hourly = electrolyser_production_profile.mul(hydrogen_baseload_price.iloc[0], axis=1) #because storage capacity is reserved and stable demand is supplied, hydrogen on average is sold for the average hydrogen price


    #Determine the storage need and costs for the electrolyser business case  
    def hydrogen_storage_need(electrolyser_production_profile):
        sum_production = sum_scenario(electrolyser_production_profile)                  #total production of the year in MWh
        baseload_production = sum_production / 8760                                     #avg production over the year in MWh/hour
        storage_need = electrolyser_production_profile * 0                              # to make a format with only zeroes
        new_row = baseload_production.loc[0].copy()                                     # Use copy() to duplicate the second row
        baseload_production2 = baseload_production._append([new_row] * 8759, ignore_index=True)  # Append the new row twice to the original DataFrame
        for index, row in electrolyser_production_profile.iterrows():
            for col in electrolyser_production_profile.columns:
                if electrolyser_production_profile.loc[index,col] > baseload_production2.loc[index,col]:
                    storage_need.loc[index,col] = electrolyser_production_profile.loc[index,col] - baseload_production2.loc[index,col]
                else:
                    storage_need.loc[index,col] = 0
        return storage_need
    
    storage_need_hourly = hydrogen_storage_need(electrolyser_production_profile)
    storage_costs_hourly = storage_need_hourly * h2_storage_tariff_list
    
    #Determine the total annual revenues and costs
    revenues_electricity_sold_market = sum_scenario(revenues_electricity_sold_market_hourly)
    revenues_electricity_sold_market_W = sum_scenario(revenues_electricity_sold_market_hourly_W)
    revenues_electricity_sold_market_S = sum_scenario(revenues_electricity_sold_market_hourly_S)
    revenues_electricity_sold_PPA = sum_scenario(revenues_electricity_sold_PPA_hourly)
    revenues_electricity_sold_PPA_W = sum_scenario(revenues_electricity_sold_PPA_hourly_W)
    revenues_electricity_sold_PPA_S = sum_scenario(revenues_electricity_sold_PPA_hourly_S)
    costs_electricity_purchased_market = sum_scenario(costs_electricity_purchased_market_hourly)
    costs_electricity_purchased_PPA = sum_scenario(costs_electricity_purchased_PPA_hourly)
    revenues_hydrogen_sold_PPA = sum_scenario(revenues_hydrogen_sold_PPA_hourly)
    storage_costs = sum_scenario(storage_costs_hourly)
    
    #Determine total volumes sold, needed for LCOE calculations in BC files (all volumes in MWh)
    electricity_sold_W = sum_scenario((electricity_sold_market+electricity_sold_PPA) * (windfarm_th_output/(windfarm_th_output+solar_th_output)))
    electricity_sold_S = sum_scenario((electricity_sold_market+electricity_sold_PPA) * (solar_th_output/(windfarm_th_output+solar_th_output)))
    hydrogen_sold = sum_scenario(electrolyser_production_profile)
    
    # print(f'electricity sold = {electricity_sold_W} MWh')
    # print(f'hydrogen sold = {hydrogen_sold} MWh')

    electricity_transported = sum_scenario(abs(electricity_sold_market - electricity_purchased_market)) # assumed that if electricity is purchased from and sold to the market at the same time, that these weight each other out

    e_purchased_from_market = sum_scenario(electricity_purchased_market)
    e_purchased_from_ppa = sum_scenario(electricity_purchased_PPA)

    #Calculations to add a table with techno-economic performance indicators
    LF_windfarm = sum_scenario(windfarm_th_output) / (windfarm_capacity*8760)
    FLH_windfarm = LF_windfarm * 8760
    LF_solar = sum_scenario(solar_th_output) / (solar_capacity*8760)
    FLH_solar = LF_solar * 8760                                                         # hours/year
    LF_electrolyser = sum_scenario((electrolyser_production_profile/efficiency_electrolyser)) / (electrolyser_capacity*8760)
    FLH_electrolyser = LF_electrolyser * 8760
    avg_electricity_price = sum_scenario(electricity_price_profiles_decade) / 8760
    avg_hydrogen_price = sum_scenario(hydrogen_price_profiles_decade) / 8760
    electricity_sold_renewables = (sum_scenario(electricity_sold_market) + sum_scenario(electricity_sold_PPA))/1E3 #MWh to GWh
    perc_e_curtailed = sum_scenario(electricity_curtailment) / (sum_scenario(windfarm_th_output) + sum_scenario(solar_th_output)) #%
    perc_e_not_purchased = sum_scenario(electricity_not_purchased_market) / sum_scenario(electrolyser_th_input)
    shortage_hours = sum_scenario(electricity_shortage_hours)
    electricity_purchased_electrolyser = (sum_scenario(electricity_purchased_market) + sum_scenario(electricity_purchased_PPA))/1E3
    hydrogen_sold_electrolyser = sum_scenario(electrolyser_production_profile) / 1E3
    hydrogen_stored = sum_scenario(storage_need_hourly) / 1E3
    perc_e_sold_market = sum_scenario(electricity_sold_market) / (electricity_sold_renewables*1E3)
    perc_e_sold_PPA = sum_scenario(electricity_sold_PPA) / (electricity_sold_renewables*1E3)
    perc_e_purchased_market = sum_scenario(electricity_purchased_market) / (electricity_purchased_electrolyser*1E3)
    perc_e_purchased_PPA = sum_scenario(electricity_purchased_PPA) / (electricity_purchased_electrolyser*1E3)
    perc_h_stored = hydrogen_stored / hydrogen_sold_electrolyser
    cap_price_E = (revenues_electricity_sold_market + revenues_electricity_sold_PPA) / (electricity_sold_renewables*1E3)
    cap_price_E_market = revenues_electricity_sold_market / sum_scenario(electricity_sold_market)
    cap_price_E_PPA = revenues_electricity_sold_PPA / sum_scenario(electricity_sold_PPA)
    cap_price_E_market_OEL = costs_electricity_purchased_market / sum_scenario(electricity_purchased_market)
    cap_price_E_OEL = (costs_electricity_purchased_market + costs_electricity_purchased_PPA) / (electricity_purchased_electrolyser*1E3)
    th_cap_price_electrolyser = sum_scenario(revenues_hydrogen_sold_PPA_hourly_without_storage) / (hydrogen_sold_electrolyser*1E3)
    revenues_wind = (revenues_electricity_sold_market_W + revenues_electricity_sold_PPA_W) / 1E6 #in MEUR/year
    revenues_solar = (revenues_electricity_sold_market_S + revenues_electricity_sold_PPA_S) / 1E6 #in MEUR/year
    margin_electrolyser = (revenues_hydrogen_sold_PPA - costs_electricity_purchased_market - costs_electricity_purchased_PPA) / 1E6 #in MEUR/year
    TE_KPI_overview_df = pd.concat([LF_windfarm, FLH_windfarm, LF_solar, FLH_solar, LF_electrolyser, FLH_electrolyser, avg_electricity_price, avg_hydrogen_price, electricity_sold_renewables, perc_e_curtailed, perc_e_not_purchased, shortage_hours, electricity_purchased_electrolyser, hydrogen_sold_electrolyser, hydrogen_stored, perc_e_sold_market, perc_e_sold_PPA, perc_e_purchased_market, perc_e_purchased_PPA, perc_h_stored, cap_price_E, cap_price_E_market, cap_price_E_PPA, cap_price_E_market_OEL, cap_price_E_OEL, th_cap_price_electrolyser, revenues_wind, revenues_solar, margin_electrolyser])
    TE_KPI_overview_df.insert(0, "KPI", ["Load factor OWF", "FLH OWF", "load factor solar", "FLH solar", "Load factor OEL", "FLH electrolyser", "Avg electricity price (EUR/MWh)", "Avg hydrogen price (EUR/MWh)", "Electricity sold by OWF (and Solar) (GWh)", "Electricity curtailment (%)", "E not purchased due to grid limitations (%)", "Min load electrolyser not met (# hours)", "Electricity purchased by OEL (GWh)","Hydrogen sold by OEL (GWh)" , "Hydrogen stored (GWh)", "Share electricity sold to market by RES (%)", "Share electricity sold via PPA by RES (%)", "Share electricity purchased from market by OEL (%)", "Share electricity purchased via by OEL (%)", "Share hydrogen stored (%)", "Capture price electricity RES (EUR/MWh)", "Capture price electricity sold to market (EUR/MWh)", "Capture price electricity sold via PPA (EUR/MWh)", "Capture price electricity purchased from market (EUR/MWh)", "Capture price total electricity purchased by OEL (EUR/MWh)", "Theoretical capture price H2 produced by OEL (EUR/MWh)", "Revenues windfarm (MEUR/y)", "Revenues solar (MEUR/y)", "Margin electrolyser (MEUR/y)" ], True)
    TE_KPI_overview_df.set_index('KPI', inplace=True) 
    # TE_KPI_overview_df = TE_KPI_overview_df.astype(float).round({'Pessimistic': 2, 'Most likely': 2, 'Optimistic': 2})         # Round will discard the additional decimals, better to use style.precision or something similar

    return FLH_electrolyser, hydrogen_baseload_price, electricity_purchased_electrolyser, e_purchased_from_market, e_purchased_from_ppa, revenues_electricity_sold_market_W, revenues_electricity_sold_PPA_W, electricity_sold_W, revenues_electricity_sold_market_S, revenues_electricity_sold_PPA_S, electricity_sold_S, costs_electricity_purchased_market, costs_electricity_purchased_PPA, revenues_hydrogen_sold_PPA, storage_costs, hydrogen_sold, electricity_transported, TE_KPI_overview_df


Execute function for 2030 and print results 

In [4]:
FLH_electrolyser_2030, h2_baseload_price_2030, E_purchased_by_EL_2030, E_purchased_by_EL_from_market_2030, E_purchased_by_EL_from_PPA_2030, rev_E_market_2030_W, rev_E_PPA_2030_W, E_sold_2030_W, rev_E_market_2030_S, rev_E_PPA_2030_S, E_sold_2030_S, costs_E_market_2030, costs_E_PPA_2030, rev_H2_PPA_2030, storage_costs_2030, H2_sold_2030, E_trans_2030, TE_KPI_overview_2030 = OWF_OEL_operation(windfarm_capacity, electrolyser_capacity, min_load_electrolyser, solar_capacity, electricity_grid_capacity, windprofiles, df_solar_profiles_NED, CF_electrolyser_profiles_2030, electricity_price_profiles_2030, hydrogen_price_profiles_2030, efficiency_electrolyser_LHV, h2_storage_tariff_list, operational_mode_electrolyser)

UHS_costs_E_2030 = TE_KPI_overview_2030.loc[['Avg electricity price (EUR/MWh)']].reset_index(drop=True)
UHS_costs_H_2030 = TE_KPI_overview_2030.loc[['Avg hydrogen price (EUR/MWh)']].reset_index(drop=True)

TE_KPI_overview_2030.style.format(precision=2)

,Pessimistic,Most likely,Optimistic
KPI,,,
Load factor OWF,0.58,0.58,0.58
FLH OWF,5112.80,5112.80,5112.80
load factor solar,nan,nan,nan
FLH solar,nan,nan,nan
Load factor OEL,0.70,0.70,0.70
FLH electrolyser,6097.26,6097.26,6097.26
Avg electricity price (EUR/MWh),61.09,51.70,47.14
Avg hydrogen price (EUR/MWh),64.83,63.71,63.44
Electricity sold by OWF (and Solar) (GWh),3578.96,3578.96,3578.96


In [5]:
E_trans_2030

,Pessimistic,Most likely,Optimistic
0,622430.0,622430.0,622430.0


Execute function for 2040 and print results

In [6]:
FLH_electrolyser_2040, h2_baseload_price_2040, E_purchased_by_EL_2040, E_purchased_by_EL_from_market_2040, E_purchased_by_EL_from_PPA_2040, rev_E_market_2040_W, rev_E_PPA_2040_W, E_sold_2040_W, rev_E_market_2040_S, rev_E_PPA_2040_S, E_sold_2040_S, costs_E_market_2040, costs_E_PPA_2040, rev_H2_PPA_2040, storage_costs_2040, H2_sold_2040, E_trans_2040, TE_KPI_overview_2040 = OWF_OEL_operation(windfarm_capacity, electrolyser_capacity, min_load_electrolyser, solar_capacity, electricity_grid_capacity, windprofiles, df_solar_profiles_NED, CF_electrolyser_profiles_2040, electricity_price_profiles_2040, hydrogen_price_profiles_2040, efficiency_electrolyser_LHV, h2_storage_tariff_list, operational_mode_electrolyser)

UHS_costs_E_2040 = TE_KPI_overview_2040.loc[['Avg electricity price (EUR/MWh)']].reset_index(drop=True)
UHS_costs_H_2040 = TE_KPI_overview_2040.loc[['Avg hydrogen price (EUR/MWh)']].reset_index(drop=True)

TE_KPI_overview_2040.style.format(precision=2)

,Pessimistic,Most likely,Optimistic
KPI,,,
Load factor OWF,0.58,0.58,0.58
FLH OWF,5112.80,5112.80,5112.80
load factor solar,nan,nan,nan
FLH solar,nan,nan,nan
Load factor OEL,0.70,0.70,0.70
FLH electrolyser,6097.26,6097.26,6097.26
Avg electricity price (EUR/MWh),141.76,108.25,96.80
Avg hydrogen price (EUR/MWh),68.04,67.02,65.64
Electricity sold by OWF (and Solar) (GWh),3578.96,3578.96,3578.96


Execute function for 2050 and print results

In [7]:
FLH_electrolyser_2050, h2_baseload_price_2050, E_purchased_by_EL_2050, E_purchased_by_EL_from_market_2050, E_purchased_by_EL_from_PPA_2050, rev_E_market_2050_W, rev_E_PPA_2050_W, E_sold_2050_W, rev_E_market_2050_S, rev_E_PPA_2050_S, E_sold_2050_S, costs_E_market_2050, costs_E_PPA_2050, rev_H2_PPA_2050, storage_costs_2050, H2_sold_2050, E_trans_2050, TE_KPI_overview_2050 = OWF_OEL_operation(windfarm_capacity, electrolyser_capacity, min_load_electrolyser, solar_capacity, electricity_grid_capacity, windprofiles, df_solar_profiles_NED, CF_electrolyser_profiles_2050, electricity_price_profiles_2050, hydrogen_price_profiles_2050, efficiency_electrolyser_LHV, h2_storage_tariff_list, operational_mode_electrolyser)

UHS_costs_E_2050 = TE_KPI_overview_2050.loc[['Avg electricity price (EUR/MWh)']].reset_index(drop=True)
UHS_costs_H_2050 = TE_KPI_overview_2050.loc[['Avg hydrogen price (EUR/MWh)']].reset_index(drop=True)

TE_KPI_overview_2050.style.format(precision=2)

,Pessimistic,Most likely,Optimistic
KPI,,,
Load factor OWF,0.58,0.58,0.58
FLH OWF,5112.80,5112.80,5112.80
load factor solar,nan,nan,nan
FLH solar,nan,nan,nan
Load factor OEL,0.70,0.70,0.70
FLH electrolyser,6097.26,6097.26,6097.26
Avg electricity price (EUR/MWh),28.55,24.16,14.04
Avg hydrogen price (EUR/MWh),46.59,43.43,27.23
Electricity sold by OWF (and Solar) (GWh),3578.96,3578.96,3578.96


Hydrogen storage (separate from storage need in operational analysis)

In [8]:
H2_stored_2030 = pd.DataFrame({'Pessimistic': [hydrogen_stored], 'Most likely': [hydrogen_stored], 'Optimistic': [hydrogen_stored]})
H2_stored_2040 = pd.DataFrame({'Pessimistic': [hydrogen_stored], 'Most likely': [hydrogen_stored], 'Optimistic': [hydrogen_stored]})
H2_stored_2050 = pd.DataFrame({'Pessimistic': [hydrogen_stored], 'Most likely': [hydrogen_stored], 'Optimistic': [hydrogen_stored]})

H2_sum_stored_2030 = pd.DataFrame({'Pessimistic': [hydrogen_in_storage_sum], 'Most likely': [hydrogen_in_storage_sum], 'Optimistic': [hydrogen_in_storage_sum]})
H2_sum_stored_2040 = pd.DataFrame({'Pessimistic': [hydrogen_in_storage_sum], 'Most likely': [hydrogen_in_storage_sum], 'Optimistic': [hydrogen_in_storage_sum]})
H2_sum_stored_2050 = pd.DataFrame({'Pessimistic': [hydrogen_in_storage_sum], 'Most likely': [hydrogen_in_storage_sum], 'Optimistic': [hydrogen_in_storage_sum]})

Save all variables into Pickle file

In [9]:
#pickle file can be used to increase computational speed
import pickle

def save_to_pickle(start_year, windfarm_capacity, solar_capacity, electricity_grid_capacity, electrolyser_capacity, lhv_h2_kwh_kg, efficiency_electrolyser_LHV,
                   storage_volume_capacity, storage_injection_capacity, storage_production_capacity, storage_type, storage_location, structure_location, #number_of_cycles, 
                   H2_stored_2030, H2_stored_2040, H2_stored_2050, H2_sum_stored_2030, H2_sum_stored_2040, H2_sum_stored_2050,
                   UHS_costs_E_2030, UHS_costs_E_2040, UHS_costs_E_2050, UHS_costs_H_2030, UHS_costs_H_2040, UHS_costs_H_2050, 
                   E_purchased_by_EL_2030, E_purchased_by_EL_2040, E_purchased_by_EL_2050,
                   E_purchased_by_EL_from_market_2030, E_purchased_by_EL_from_market_2040, E_purchased_by_EL_from_market_2050, 
                   E_purchased_by_EL_from_PPA_2030, E_purchased_by_EL_from_PPA_2040, E_purchased_by_EL_from_PPA_2050,
                   h2_baseload_price_2030, h2_baseload_price_2040, h2_baseload_price_2050,
                   rev_E_market_2030_W, rev_E_PPA_2030_W, E_sold_2030_W, rev_E_market_2030_S, rev_E_PPA_2030_S, E_sold_2030_S, costs_E_market_2030, costs_E_PPA_2030, rev_H2_PPA_2030, storage_costs_2030, H2_sold_2030, E_trans_2030,
                   rev_E_market_2040_W, rev_E_PPA_2040_W, E_sold_2040_W, rev_E_market_2040_S, rev_E_PPA_2040_S, E_sold_2040_S, costs_E_market_2040, costs_E_PPA_2040, rev_H2_PPA_2040, storage_costs_2040, H2_sold_2040, E_trans_2040,
                   rev_E_market_2050_W, rev_E_PPA_2050_W, E_sold_2050_W, rev_E_market_2050_S, rev_E_PPA_2050_S, E_sold_2050_S, costs_E_market_2050, costs_E_PPA_2050, rev_H2_PPA_2050, storage_costs_2050, H2_sold_2050, E_trans_2050,
                   FLH_electrolyser_2030, FLH_electrolyser_2040, FLH_electrolyser_2050):

    # Define a filename for the pickle file
    filename = 'Operation_Analysis_variables.pkl'

    # Choose the specific variables to be saved
    variables_to_save = {
        'start_year': start_year,
        'windfarm_capacity': windfarm_capacity,  
        'solar_capacity': solar_capacity,  
        'electricity_grid_capacity': electricity_grid_capacity,
        'electrolyser_capacity': electrolyser_capacity,
        'lhv_h2_kwh_kg': lhv_h2_kwh_kg,
        'efficiency_electrolyser_LHV': efficiency_electrolyser_LHV,           

        'storage_volume_capacity': storage_volume_capacity,
        'storage_injection_capacity': storage_injection_capacity,
        'storage_production_capacity': storage_production_capacity,
        'storage_type': storage_type,
        'storage_location': storage_location,
        'structure_location': structure_location,
        #'number_of_cycles': number_of_cycles,
        'H2_stored_2030': H2_stored_2030,
        'H2_stored_2040': H2_stored_2040,
        'H2_stored_2050': H2_stored_2050,
        'H2_sum_stored_2030': H2_sum_stored_2030,
        'H2_sum_stored_2040': H2_sum_stored_2040,
        'H2_sum_stored_2050': H2_sum_stored_2050,
        
        'UHS_costs_E_2030': UHS_costs_E_2030,
        'UHS_costs_E_2040': UHS_costs_E_2040,
        'UHS_costs_E_2050': UHS_costs_E_2050,
        'UHS_costs_H_2030': UHS_costs_H_2030,
        'UHS_costs_H_2040': UHS_costs_H_2040,
        'UHS_costs_H_2050': UHS_costs_H_2050,

        'E_purchased_by_EL_2030': E_purchased_by_EL_2030,
        'E_purchased_by_EL_2040': E_purchased_by_EL_2040,
        'E_purchased_by_EL_2050': E_purchased_by_EL_2050,

        'E_purchased_by_EL_from_market_2030': E_purchased_by_EL_from_market_2030,
        'E_purchased_by_EL_from_market_2040': E_purchased_by_EL_from_market_2040,
        'E_purchased_by_EL_from_market_2050': E_purchased_by_EL_from_market_2050,
        'E_purchased_by_EL_from_PPA_2030': E_purchased_by_EL_from_PPA_2030,
        'E_purchased_by_EL_from_PPA_2040': E_purchased_by_EL_from_PPA_2040,
        'E_purchased_by_EL_from_PPA_2050': E_purchased_by_EL_from_PPA_2050,

        'h2_baseload_price_2030': h2_baseload_price_2030,
        'h2_baseload_price_2040': h2_baseload_price_2040,
        'h2_baseload_price_2050': h2_baseload_price_2050,

        'rev_E_market_2030_W': rev_E_market_2030_W, 
        'rev_E_PPA_2030_W': rev_E_PPA_2030_W,
        'E_sold_2030_W': E_sold_2030_W,
        'rev_E_market_2030_S': rev_E_market_2030_S, 
        'rev_E_PPA_2030_S': rev_E_PPA_2030_S,
        'E_sold_2030_S': E_sold_2030_S,
        'costs_E_market_2030': costs_E_market_2030, 
        'costs_E_PPA_2030': costs_E_PPA_2030, 
        'rev_H2_PPA_2030': rev_H2_PPA_2030, 
        'storage_costs_2030': storage_costs_2030,
        'H2_sold_2030': H2_sold_2030,
        'E_trans_2030': E_trans_2030,  
        
        'rev_E_market_2040_W': rev_E_market_2040_W, 
        'rev_E_PPA_2040_W': rev_E_PPA_2040_W,
        'E_sold_2040_W': E_sold_2040_W,
        'rev_E_market_2040_S': rev_E_market_2040_S, 
        'rev_E_PPA_2040_S': rev_E_PPA_2040_S,
        'E_sold_2040_S': E_sold_2040_S,
        'costs_E_market_2040': costs_E_market_2040, 
        'costs_E_PPA_2040': costs_E_PPA_2040, 
        'rev_H2_PPA_2040': rev_H2_PPA_2040, 
        'storage_costs_2040': storage_costs_2040,
        'H2_sold_2040': H2_sold_2040, 
        'E_trans_2040': E_trans_2040,

        'rev_E_market_2050_W': rev_E_market_2050_W, 
        'rev_E_PPA_2050_W': rev_E_PPA_2050_W,
        'E_sold_2050_W': E_sold_2050_W,
        'rev_E_market_2050_S': rev_E_market_2050_S, 
        'rev_E_PPA_2050_S': rev_E_PPA_2050_S,
        'E_sold_2050_S': E_sold_2050_S,
        'costs_E_market_2050': costs_E_market_2050, 
        'costs_E_PPA_2050': costs_E_PPA_2050, 
        'rev_H2_PPA_2050': rev_H2_PPA_2050, 
        'storage_costs_2050': storage_costs_2050,
        'H2_sold_2050': H2_sold_2050, 
        'E_trans_2050': E_trans_2050,

        'FLH_electrolyser_2030': FLH_electrolyser_2030,
        'FLH_electrolyser_2040': FLH_electrolyser_2040,
        'FLH_electrolyser_2050': FLH_electrolyser_2050,
    }

    #### Uncomment to save all variables. However, this gives some issues when using them in other files
    #variables_to_save = {}

    #for k, v in globals().items():
        # Controleer of de waarde geen functie is en niet van het type 'types.FunctionType'
    #    if not isinstance(v, types.FunctionType):
    #        try:
                # Probeer de variabele te serialiseren om te controleren of hij picklebaar is
    #            pickle.dumps(v)
    #            variables_to_save[k] = v
    #        except (pickle.PicklingError, AttributeError, TypeError) as e:
    #            # Toon welke variabele niet picklebaar is en de reden
    #           print(f"Skipping non-pickleable variable: {k} - Error: {e}")

    #Save variables to Pickle file
    with open(filename, 'wb') as f:
        pickle.dump(variables_to_save, f)

    print(f"All variables saved to {filename}")

save_to_pickle(start_year, windfarm_capacity, solar_capacity, electricity_grid_capacity, electrolyser_capacity, lhv_h2_kwh_kg, efficiency_electrolyser_LHV,
               storage_volume_capacity, storage_injection_capacity, storage_production_capacity, storage_type, storage_location, structure_location, #number_of_cycles,
               H2_stored_2030, H2_stored_2040, H2_stored_2050, H2_sum_stored_2030, H2_sum_stored_2040, H2_sum_stored_2050,
               UHS_costs_E_2030, UHS_costs_E_2040, UHS_costs_E_2050, UHS_costs_H_2030, UHS_costs_H_2040, UHS_costs_H_2050,
               E_purchased_by_EL_2030, E_purchased_by_EL_2040, E_purchased_by_EL_2050, 
               E_purchased_by_EL_from_market_2030, E_purchased_by_EL_from_market_2040, E_purchased_by_EL_from_market_2050, 
               E_purchased_by_EL_from_PPA_2030, E_purchased_by_EL_from_PPA_2040, E_purchased_by_EL_from_PPA_2050,
               h2_baseload_price_2030, h2_baseload_price_2040, h2_baseload_price_2050,
               rev_E_market_2030_W, rev_E_PPA_2030_W, E_sold_2030_W, rev_E_market_2030_S, rev_E_PPA_2030_S, E_sold_2030_S, costs_E_market_2030, costs_E_PPA_2030, rev_H2_PPA_2030, storage_costs_2030, H2_sold_2030, E_trans_2030,
               rev_E_market_2040_W, rev_E_PPA_2040_W, E_sold_2040_W, rev_E_market_2040_S, rev_E_PPA_2040_S, E_sold_2040_S, costs_E_market_2040, costs_E_PPA_2040, rev_H2_PPA_2040, storage_costs_2040, H2_sold_2040, E_trans_2040,
               rev_E_market_2050_W, rev_E_PPA_2050_W, E_sold_2050_W, rev_E_market_2050_S, rev_E_PPA_2050_S, E_sold_2050_S, costs_E_market_2050, costs_E_PPA_2050, rev_H2_PPA_2050, storage_costs_2050, H2_sold_2050, E_trans_2050,
               FLH_electrolyser_2030, FLH_electrolyser_2040, FLH_electrolyser_2050
               )

All variables saved to Operation_Analysis_variables.pkl
